In [5]:
from importlib.metadata import version, PackageNotFoundError

# Check modular SageMaker V3 packages
packages = [
    "sagemaker-core",
    "sagemaker-train",
    "sagemaker-serve",
    "sagemaker-mlops",
]

for pkg in packages:
    try:
        print(f"{pkg}: {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg}: NOT INSTALLED")

print("\nTesting Pipeline imports...")

from sagemaker.core.workflow.pipeline_context import PipelineSession

from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.mlops.workflow.steps import (
    ProcessingStep,
    TrainingStep,
)
from sagemaker.mlops.workflow.model_step import ModelStep

print("Pipeline imports successful ✅")

sagemaker-core: 2.20.0
sagemaker-train: 1.14.0
sagemaker-serve: 1.12.0
sagemaker-mlops: 1.12.0

Testing Pipeline imports...
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Pipeline imports successful ✅


In [6]:
pipeline_session = PipelineSession()

print("PipelineSession created ✅")
print("Region:", pipeline_session.boto_region_name)

PipelineSession created ✅
Region: ap-south-1


In [7]:
from pathlib import Path

from sagemaker.core.helper.session_helper import get_execution_role
from sagemaker.core import image_uris
from sagemaker.core.processing import FrameworkProcessor
from sagemaker.core.shapes import (
    ProcessingInput,
    ProcessingS3Input,
    ProcessingOutput,
    ProcessingS3Output,
)
from sagemaker.mlops.workflow.steps import ProcessingStep


# ---------------------------------------------------------
# Project root
# ---------------------------------------------------------

ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

print("Project root:", ROOT)


# ---------------------------------------------------------
# AWS configuration
# ---------------------------------------------------------

region = pipeline_session.boto_region_name
role = get_execution_role()

bucket = "krushang-beverage-ml-2026"

raw_data_uri = (
    f"s3://{bucket}/raw/survey_results.csv"
)

pipeline_processed_uri = (
    f"s3://{bucket}/pipeline/processed"
)

print("Region:", region)
print("Role:", role)
print("Raw data:", raw_data_uri)
print("Pipeline output:", pipeline_processed_uri)

Project root: /home/sagemaker-user/Beverage-Price-Prediction-AWS
Region: ap-south-1
Role: arn:aws:iam::812224290846:role/service-role/AmazonSageMaker-ExecutionRole-20260902T184223
Raw data: s3://krushang-beverage-ml-2026/raw/survey_results.csv
Pipeline output: s3://krushang-beverage-ml-2026/pipeline/processed


In [8]:
sklearn_image = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.4-2",
    py_version="py3",
    instance_type="ml.t3.medium",
)

print("Processing image:")
print(sklearn_image)

Processing image:
720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-cpu-py3


In [9]:
preprocessing_dir = ROOT / "src" / "preprocessing"

processor = FrameworkProcessor(
    image_uri=sklearn_image,
    role=role,
    instance_count=1,
    instance_type="ml.t3.medium",
    command=["python3"],
    base_job_name="beverage-pipeline-preprocess",
    sagemaker_session=pipeline_session,
)

print("Processor configured ✅")
print("Source directory:", preprocessing_dir)

Processor configured ✅
Source directory: /home/sagemaker-user/Beverage-Price-Prediction-AWS/src/preprocessing


In [10]:
process_args = processor.run(
    code=str(preprocessing_dir / "preprocess.py"),
    source_dir=str(preprocessing_dir),

    inputs=[
        ProcessingInput(
            input_name="raw-data",
            s3_input=ProcessingS3Input(
                s3_uri=raw_data_uri,
                local_path="/opt/ml/processing/input",
                s3_data_type="S3Prefix",
            ),
        )
    ],

    outputs=[
        ProcessingOutput(
            output_name="processed-data",
            s3_output=ProcessingS3Output(
                s3_uri=pipeline_processed_uri,
                local_path="/opt/ml/processing/output",
                s3_upload_mode="EndOfJob",
            ),
        )
    ],
)


step_process = ProcessingStep(
    name="BeveragePreprocessing",
    step_args=process_args,
)

print("ProcessingStep created ✅")
print("Step name:", step_process.name)

[09/05/26 09:50:11] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=8699570;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=8699571;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#304\304]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


ProcessingStep created ✅
Step name: BeveragePreprocessing


In [11]:
train_script = ROOT / "src" / "training" / "train.py"

print("Training script:", train_script)
print("=" * 80)

with open(train_script, "r") as f:
    lines = f.readlines()

keywords = [
    "argparse",
    "add_argument",
    "SM_CHANNEL",
    "SM_MODEL_DIR",
    "SM_OUTPUT",
    "best_params",
    "processed",
    "train",
    "csv",
]

for i, line in enumerate(lines, start=1):
    if any(keyword in line for keyword in keywords):
        print(f"{i:03d}: {line.rstrip()}")

Training script: /home/sagemaker-user/Beverage-Price-Prediction-AWS/src/training/train.py
049:     training_dir = os.environ["SM_CHANNEL_TRAINING"]
050:     config_dir = os.environ["SM_CHANNEL_CONFIG"]
051:     model_dir = os.environ["SM_MODEL_DIR"]
058:         training_dir,
059:         ".csv"
067:     df = pd.read_csv(data_path)
070:         best_params = json.load(f)
105:         **best_params,
153:         "training_rows": int(len(df)),
155:         "hyperparameters": best_params,
158:         "trained_at_utc": datetime.now(
171:             "training_metadata.json"
181:     print("Production model training complete.")


In [12]:
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import (
    SourceCode,
    Compute,
    InputData,
)

training_dir = ROOT / "src" / "training"

best_params_uri = (
    f"s3://{bucket}/evaluation/xgboost_best_params.json"
)

# Use the same sklearn container family as our successful managed training
training_image = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.4-2",
    py_version="py3",
    instance_type="ml.m5.large",
)

source_code = SourceCode(
    source_dir=str(training_dir),
    entry_script="train.py",
    requirements="requirements.txt",
)

compute = Compute(
    instance_type="ml.m5.large",
    instance_count=1,
)

trainer = ModelTrainer(
    training_image=training_image,
    role=role,
    source_code=source_code,
    compute=compute,
    base_job_name="beverage-pipeline-training",
    sagemaker_session=pipeline_session,
)

print("Training configuration created ✅")
print("Training image:", training_image)
print("Best params:", best_params_uri)

[09/05/26 09:55:15] INFO     StoppingCondition not provided. Using default:                         ]8;id=8699578;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=8699579;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     OutputDataConfig not provided. Using default:                          ]8;id=8699585;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=8699586;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#153\153]8;;\
                             s3_output_path='s3://sagemaker-ap-south-1-812224290846/beverage-pipeli                
                             ne-training' kms_key_id=None compression_type='GZIP'                                  

                    INFO     Training image URI:                                               ]8;id=8699593;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=8699594;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-le                     
                             arn:1.4-2-cpu-py3                                                                     

Training configuration created ✅
Training image: 720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-cpu-py3
Best params: s3://krushang-beverage-ml-2026/evaluation/xgboost_best_params.json


In [14]:
processed_data_uri = (
    step_process
    .properties
    .ProcessingOutputConfig
    .Outputs["processed-data"]
    .S3Output
    .S3Uri
)

training_input = InputData(
    channel_name="training",
    data_source=processed_data_uri,
)

config_input = InputData(
    channel_name="config",
    data_source=best_params_uri,
)

print("Training channel expression:")
print(processed_data_uri.expr)

print("\nConfig channel:")
print(best_params_uri)

print("\nTraining inputs configured ✅")

Training channel expression:
{'Get': "Steps.BeveragePreprocessing.ProcessingOutputConfig.Outputs['processed-data'].S3Output.S3Uri"}

Config channel:
s3://krushang-beverage-ml-2026/evaluation/xgboost_best_params.json

Training inputs configured ✅


In [15]:
train_args = trainer.train(
    input_data_config=[
        training_input,
        config_input,
    ]
)

step_train = TrainingStep(
    name="BeverageXGBoostTraining",
    step_args=train_args,
)

print("TrainingStep created ✅")
print("Step name:", step_train.name)

TrainingStep created ✅
Step name: BeverageXGBoostTraining


/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [16]:
from pathlib import Path

pipeline_src = ROOT / "src" / "pipeline"
pipeline_src.mkdir(parents=True, exist_ok=True)

split_script = pipeline_src / "split_data.py"

split_script.write_text(r'''
import os
import json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split


INPUT_DIR = "/opt/ml/processing/input"
TRAIN_DIR = "/opt/ml/processing/train"
TEST_DIR = "/opt/ml/processing/test"
METADATA_DIR = "/opt/ml/processing/metadata"

TARGET = "price_range"
TEST_SIZE = 0.25
RANDOM_STATE = 42


def main():

    input_files = list(Path(INPUT_DIR).rglob("*.csv"))

    if not input_files:
        raise FileNotFoundError(
            f"No CSV found under {INPUT_DIR}"
        )

    input_file = input_files[0]

    print("Input:", input_file)

    df = pd.read_csv(input_file)

    if TARGET not in df.columns:
        raise ValueError(
            f"Target column '{TARGET}' not found."
        )

    train_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=df[TARGET],
    )

    os.makedirs(TRAIN_DIR, exist_ok=True)
    os.makedirs(TEST_DIR, exist_ok=True)
    os.makedirs(METADATA_DIR, exist_ok=True)

    train_path = os.path.join(
        TRAIN_DIR,
        "train.csv"
    )

    test_path = os.path.join(
        TEST_DIR,
        "test.csv"
    )

    train_df.to_csv(
        train_path,
        index=False
    )

    test_df.to_csv(
        test_path,
        index=False
    )

    metadata = {
        "total_rows": int(len(df)),
        "training_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "stratified_by": TARGET,
    }

    with open(
        os.path.join(
            METADATA_DIR,
            "split_metadata.json"
        ),
        "w"
    ) as f:
        json.dump(metadata, f, indent=4)

    print(json.dumps(metadata, indent=4))
    print("Train:", train_path)
    print("Test :", test_path)


if __name__ == "__main__":
    main()
''')

print("Created:", split_script)

Created: /home/sagemaker-user/Beverage-Price-Prediction-AWS/src/pipeline/split_data.py


In [17]:
split_processor = FrameworkProcessor(
    image_uri=sklearn_image,
    role=role,
    instance_count=1,
    instance_type="ml.t3.medium",
    command=["python3"],
    base_job_name="beverage-pipeline-split",
    sagemaker_session=pipeline_session,
)


split_args = split_processor.run(

    code=str(split_script),
    source_dir=str(pipeline_src),

    inputs=[
        ProcessingInput(
            input_name="cleaned-data",
            s3_input=ProcessingS3Input(
                s3_uri=processed_data_uri,
                local_path="/opt/ml/processing/input",
                s3_data_type="S3Prefix",
            ),
        )
    ],

    outputs=[

        ProcessingOutput(
            output_name="train-data",
            s3_output=ProcessingS3Output(
                s3_uri=f"s3://{bucket}/pipeline/split/train",
                local_path="/opt/ml/processing/train",
                s3_upload_mode="EndOfJob",
            ),
        ),

        ProcessingOutput(
            output_name="test-data",
            s3_output=ProcessingS3Output(
                s3_uri=f"s3://{bucket}/pipeline/split/test",
                local_path="/opt/ml/processing/test",
                s3_upload_mode="EndOfJob",
            ),
        ),

        ProcessingOutput(
            output_name="split-metadata",
            s3_output=ProcessingS3Output(
                s3_uri=f"s3://{bucket}/pipeline/split/metadata",
                local_path="/opt/ml/processing/metadata",
                s3_upload_mode="EndOfJob",
            ),
        ),
    ],
)


step_split = ProcessingStep(
    name="BeverageTrainTestSplit",
    step_args=split_args,
)

print("Split step created ✅")
print("Step:", step_split.name)

Split step created ✅
Step: BeverageTrainTestSplit


/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [18]:
pipeline_train_uri = (
    step_split
    .properties
    .ProcessingOutputConfig
    .Outputs["train-data"]
    .S3Output
    .S3Uri
)

pipeline_test_uri = (
    step_split
    .properties
    .ProcessingOutputConfig
    .Outputs["test-data"]
    .S3Output
    .S3Uri
)


training_input = InputData(
    channel_name="training",
    data_source=pipeline_train_uri,
)

config_input = InputData(
    channel_name="config",
    data_source=best_params_uri,
)


train_args = trainer.train(
    input_data_config=[
        training_input,
        config_input,
    ]
)


# Replace the earlier TrainingStep definition
step_train = TrainingStep(
    name="BeverageXGBoostTraining",
    step_args=train_args,
)


print("Leakage-safe TrainingStep recreated ✅")

print("\nTraining source:")
print(pipeline_train_uri.expr)

print("\nEvaluation source:")
print(pipeline_test_uri.expr)

Leakage-safe TrainingStep recreated ✅

Training source:
{'Get': "Steps.BeverageTrainTestSplit.ProcessingOutputConfig.Outputs['train-data'].S3Output.S3Uri"}

Evaluation source:
{'Get': "Steps.BeverageTrainTestSplit.ProcessingOutputConfig.Outputs['test-data'].S3Output.S3Uri"}


/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [19]:
from sagemaker.train.configs import SourceCode, Compute, InputData
from sagemaker.train import ModelTrainer


source_code = SourceCode(
    source_dir=str(ROOT / "src"),
    entry_script="training/train.py",
    requirements="training/requirements.txt",
)

compute = Compute(
    instance_type="ml.m5.large",
    instance_count=1,
)

trainer = ModelTrainer(
    training_image=training_image,
    role=role,
    source_code=source_code,
    compute=compute,
    base_job_name="beverage-pipeline-training",
    sagemaker_session=pipeline_session,
)


training_input = InputData(
    channel_name="training",
    data_source=pipeline_train_uri,
)

config_input = InputData(
    channel_name="config",
    data_source=best_params_uri,
)


train_args = trainer.train(
    input_data_config=[
        training_input,
        config_input,
    ]
)

step_train = TrainingStep(
    name="BeverageXGBoostTraining",
    step_args=train_args,
)

print("TrainingStep corrected ✅")
print("Source bundle:", ROOT / "src")
print("Entry script: training/train.py")

[09/05/26 10:07:08] INFO     StoppingCondition not provided. Using default:                         ]8;id=8699599;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=8699600;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     OutputDataConfig not provided. Using default:                          ]8;id=8699605;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=8699606;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#153\153]8;;\
                             s3_output_path='s3://sagemaker-ap-south-1-812224290846/beverage-pipeli                
                             ne-training' kms_key_id=None compression_type='GZIP'                                  

                    INFO     Training image URI:                                               ]8;id=8699611;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=8699612;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-le                     
                             arn:1.4-2-cpu-py3                                                                     

TrainingStep corrected ✅
Source bundle: /home/sagemaker-user/Beverage-Price-Prediction-AWS/src
Entry script: training/train.py


/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [20]:
evaluation_dir = ROOT / "src" / "evaluation"
evaluation_dir.mkdir(parents=True, exist_ok=True)

evaluate_script = evaluation_dir / "evaluate.py"

evaluate_script.write_text(r'''
import os
import json
import tarfile
from pathlib import Path

import joblib
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from preprocessing.feature_utils import transform_features


MODEL_DIR = "/opt/ml/processing/model"
TEST_DIR = "/opt/ml/processing/test"
OUTPUT_DIR = "/opt/ml/processing/evaluation"

TARGET = "price_range"


def find_file(root, pattern):
    files = list(Path(root).rglob(pattern))

    if not files:
        raise FileNotFoundError(
            f"Could not find {pattern} under {root}"
        )

    return files[0]


def main():

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True,
    )

    # --------------------------------------------------
    # Load trained model artifact
    # --------------------------------------------------

    model_tar = find_file(
        MODEL_DIR,
        "model.tar.gz",
    )

    extracted_dir = "/tmp/beverage-model"

    os.makedirs(
        extracted_dir,
        exist_ok=True,
    )

    with tarfile.open(
        model_tar,
        "r:gz",
    ) as tar:
        tar.extractall(extracted_dir)


    bundle_path = find_file(
        extracted_dir,
        "model_bundle.joblib",
    )

    bundle = joblib.load(
        bundle_path
    )

    model = bundle["model"]
    preprocessing = bundle["preprocessing"]
    price_map = bundle["price_map"]


    # --------------------------------------------------
    # Load untouched test data
    # --------------------------------------------------

    test_file = find_file(
        TEST_DIR,
        "*.csv",
    )

    test_df = pd.read_csv(
        test_file
    )

    if TARGET not in test_df.columns:
        raise ValueError(
            f"Target column {TARGET} missing."
        )


    # --------------------------------------------------
    # Prepare target
    # --------------------------------------------------

    y_true = (
        test_df[TARGET]
        .map(price_map)
        .astype(int)
    )


    # --------------------------------------------------
    # Prepare model features
    # --------------------------------------------------

    X_raw = test_df.drop(
        columns=[
            TARGET,
            "respondent_id",
        ],
        errors="ignore",
    )

    X_test = transform_features(
        X_raw,
        preprocessing,
    )


    # --------------------------------------------------
    # Predict
    # --------------------------------------------------

    y_pred = model.predict(
        X_test
    ).astype(int)


    # --------------------------------------------------
    # Metrics
    # --------------------------------------------------

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    macro_precision = precision_score(
        y_true,
        y_pred,
        average="macro",
    )

    macro_recall = recall_score(
        y_true,
        y_pred,
        average="macro",
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
    )

    ordinal_mae = np.abs(
        y_true.to_numpy() - y_pred
    ).mean()


    report = {
        "metrics": {
            "accuracy": float(accuracy),
            "macro_precision": float(macro_precision),
            "macro_recall": float(macro_recall),
            "macro_f1": float(macro_f1),
            "weighted_f1": float(weighted_f1),
            "ordinal_mae": float(ordinal_mae),
        },
        "test_rows": int(len(test_df)),
        "feature_count": int(X_test.shape[1]),
    }


    # --------------------------------------------------
    # Save evaluation report
    # --------------------------------------------------

    evaluation_path = os.path.join(
        OUTPUT_DIR,
        "evaluation.json",
    )

    with open(
        evaluation_path,
        "w",
    ) as f:
        json.dump(
            report,
            f,
            indent=4,
        )


    # --------------------------------------------------
    # Save confusion matrix
    # --------------------------------------------------

    cm = confusion_matrix(
        y_true,
        y_pred,
    )

    cm_df = pd.DataFrame(
        cm
    )

    cm_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "confusion_matrix.csv",
        ),
        index=False,
    )


    print(
        json.dumps(
            report,
            indent=4,
        )
    )

    print(
        "Evaluation complete ✅"
    )


if __name__ == "__main__":
    main()
''')

print("Created:", evaluate_script)

Created: /home/sagemaker-user/Beverage-Price-Prediction-AWS/src/evaluation/evaluate.py


In [21]:
requirements_path = (
    evaluation_dir /
    "requirements.txt"
)

requirements_path.write_text(
"""numpy==1.26.4
pandas==2.3.3
scikit-learn==1.7.2
xgboost-cpu==2.1.4
joblib>=1.3,<2
"""
)

print("Evaluation requirements created ✅")

Evaluation requirements created ✅


In [22]:
from sagemaker.core.workflow.properties import PropertyFile


evaluation_processor = FrameworkProcessor(
    image_uri=sklearn_image,
    role=role,
    instance_count=1,
    instance_type="ml.t3.medium",
    command=["python3"],
    base_job_name="beverage-pipeline-evaluation",
    sagemaker_session=pipeline_session,
)


evaluation_report = PropertyFile(
    name="BeverageEvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)


evaluation_args = evaluation_processor.run(

    code=str(
        evaluation_dir /
        "evaluate.py"
    ),

    source_dir=str(
        evaluation_dir
    ),

    dependencies=[
        str(
            ROOT /
            "src" /
            "preprocessing"
        )
    ],

    requirements="requirements.txt",

    inputs=[

        # Model from TrainingStep
        ProcessingInput(
            input_name="model",
            s3_input=ProcessingS3Input(
                s3_uri=(
                    step_train
                    .properties
                    .ModelArtifacts
                    .S3ModelArtifacts
                ),
                local_path="/opt/ml/processing/model",
                s3_data_type="S3Prefix",
            ),
        ),

        # Untouched test data
        ProcessingInput(
            input_name="test-data",
            s3_input=ProcessingS3Input(
                s3_uri=pipeline_test_uri,
                local_path="/opt/ml/processing/test",
                s3_data_type="S3Prefix",
            ),
        ),
    ],

    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            s3_output=ProcessingS3Output(
                s3_uri=(
                    f"s3://{bucket}/"
                    "pipeline/evaluation"
                ),
                local_path="/opt/ml/processing/evaluation",
                s3_upload_mode="EndOfJob",
            ),
        )
    ],
)


step_evaluate = ProcessingStep(
    name="BeverageModelEvaluation",
    step_args=evaluation_args,
    property_files=[
        evaluation_report
    ],
)


print("EvaluationStep created ✅")
print("Step:", step_evaluate.name)

EvaluationStep created ✅
Step: BeverageModelEvaluation


/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [23]:
# Full cleaned dataset from preprocessing
production_training_input = InputData(
    channel_name="training",
    data_source=processed_data_uri,
)

production_config_input = InputData(
    channel_name="config",
    data_source=best_params_uri,
)


production_train_args = trainer.train(
    input_data_config=[
        production_training_input,
        production_config_input,
    ]
)


step_refit = TrainingStep(
    name="BeverageProductionRefit",
    step_args=production_train_args,
)

print("Production refit step created ✅")

Production refit step created ✅


/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [24]:
import shutil
from pathlib import Path

registry_source = Path(
    "/tmp/beverage-registry-source"
)

if registry_source.exists():
    shutil.rmtree(registry_source)

registry_source.mkdir(
    parents=True,
    exist_ok=True,
)


# Inference entry point
shutil.copy(
    ROOT / "src/inference/inference.py",
    registry_source / "inference.py",
)

# Inference dependencies
shutil.copy(
    ROOT / "src/inference/requirements.txt",
    registry_source / "requirements.txt",
)

# setup.py
shutil.copy(
    ROOT / "src/inference/setup.py",
    registry_source / "setup.py",
)

# Shared preprocessing package
shutil.copytree(
    ROOT / "src/preprocessing",
    registry_source / "preprocessing",
)


print("Registry package prepared ✅")

for path in sorted(
    registry_source.rglob("*")
):
    if path.is_file():
        print(
            path.relative_to(
                registry_source
            )
        )

Registry package prepared ✅
inference.py
preprocessing/__init__.py
preprocessing/__pycache__/__init__.cpython-312.pyc
preprocessing/__pycache__/feature_utils.cpython-312.pyc
preprocessing/feature_utils.py
preprocessing/preprocess.py
requirements.txt
setup.py


In [25]:
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.core.training.configs import SourceCode
from sagemaker.core.model_metrics import (
    ModelMetrics,
    MetricsSource,
)
from sagemaker.core.workflow.functions import Join


MODEL_PACKAGE_GROUP = (
    "beverage-price-prediction-xgboost"
)


# ---------------------------------------------------------
# Dynamic evaluation report URI
# ---------------------------------------------------------

evaluation_output_uri = (
    step_evaluate
    .properties
    .ProcessingOutputConfig
    .Outputs["evaluation"]
    .S3Output
    .S3Uri
)


evaluation_json_uri = Join(
    on="/",
    values=[
        evaluation_output_uri,
        "evaluation.json",
    ],
)


model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        content_type="application/json",
        s3_uri=evaluation_json_uri,
    )
)


# ---------------------------------------------------------
# Inference source
# ---------------------------------------------------------

inference_source = SourceCode(
    source_dir=str(registry_source),
    entry_script="inference.py",
    requirements="requirements.txt",
)


# ---------------------------------------------------------
# Build registry model
# ---------------------------------------------------------

model_builder = ModelBuilder(

    # IMPORTANT:
    # Register the final all-data refit,
    # not the candidate model.
    s3_model_data_url=(
        step_refit
        .properties
        .ModelArtifacts
        .S3ModelArtifacts
    ),

    image_uri=sklearn_image,

    source_code=inference_source,

    role_arn=role,

    sagemaker_session=pipeline_session,

    env_vars={
        "PIP_BREAK_SYSTEM_PACKAGES": "1",

        "PIP_TARGET":
            "/opt/ml/model/code/site-packages",

        "PYTHONPATH":
            "/opt/ml/model/code:"
            "/opt/ml/model/code/site-packages",

        "SAGEMAKER_PROGRAM":
            "inference.py",

        "SAGEMAKER_SUBMIT_DIRECTORY":
            "/opt/ml/model/code",
    },
)


register_args = model_builder.register(

    model_package_group_name=(
        MODEL_PACKAGE_GROUP
    ),

    content_types=[
        "application/json"
    ],

    response_types=[
        "application/json"
    ],

    model_metrics=model_metrics,

    # Quality gate passes first,
    # but human approval remains separate.
    approval_status=(
        "PendingManualApproval"
    ),

    description=(
        "XGBoost beverage price-range model "
        "created by automated SageMaker Pipeline."
    ),
)


step_register = ModelStep(
    name="RegisterBeverageModel",
    step_args=register_args,
)

print("Model Registry step created ✅")

[09/05/26 10:11:06] DEBUG    Auto-detecting optimal instance type for model...           ]8;id=8699619;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=8699620;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#340\340]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=8699626;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=8699627;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#374\374]8;;\

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


[09/05/26 10:11:07] WARNING  No models to repack                                                  ]8;id=8699634;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py\model_step.py]8;;\:]8;id=8699635;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py#218\218]8;;\

Model Registry step created ✅


In [26]:
from sagemaker.core.workflow.functions import JsonGet
from sagemaker.core.workflow.conditions import (
    ConditionGreaterThanOrEqualTo,
)
from sagemaker.mlops.workflow.condition_step import (
    ConditionStep,
)


macro_f1 = JsonGet(
    step_name=step_evaluate.name,
    property_file=evaluation_report,
    json_path="metrics.macro_f1",
)


f1_condition = (
    ConditionGreaterThanOrEqualTo(
        left=macro_f1,
        right=0.90,
    )
)


step_condition = ConditionStep(

    name="CheckModelQuality",

    conditions=[
        f1_condition
    ],

    if_steps=[
        step_refit,
        step_register,
    ],

    else_steps=[],
)


print("Quality gate created ✅")
print("Threshold: macro_f1 >= 0.90")

Quality gate created ✅
Threshold: macro_f1 >= 0.90


In [27]:
pipeline = Pipeline(

    name=(
        "beverage-price-prediction-pipeline"
    ),

    steps=[
        step_process,
        step_split,
        step_train,
        step_evaluate,
        step_condition,
    ],

    sagemaker_session=pipeline_session,
)


definition = pipeline.definition()

print("Pipeline definition created ✅")
print(
    "Definition size:",
    len(definition),
    "characters"
)

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


[09/05/26 10:11:38] WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=8699642;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8699643;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=8699648;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8699649;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[09/05/26 10:11:39] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=8699654;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8699655;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

[09/05/26 10:11:40] WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=8699660;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8699661;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[09/05/26 10:11:41] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=8699666;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8699667;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'CertifyForMarketplace' from the pipeline definition     ]8;id=8699673;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py\model_step.py]8;;\:]8;id=8699674;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py#195\195]8;;\
                             since it will be overridden in pipeline execution time.                               

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=8699679;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8699680;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

Pipeline definition created ✅
Definition size: 11642 characters


In [28]:
response = pipeline.upsert(
    role_arn=role
)

print("Pipeline upserted ✅")
print(
    response.get(
        "PipelineArn"
    )
)

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


[09/05/26 10:11:48] WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=8699685;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8699686;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=8699691;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8699692;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[09/05/26 10:11:49] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=8699697;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8699698;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

[09/05/26 10:11:50] WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=8699703;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8699704;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[09/05/26 10:11:51] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=8699709;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8699710;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=8699715;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8699716;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

Pipeline upserted ✅
arn:aws:sagemaker:ap-south-1:812224290846:pipeline/beverage-price-prediction-pipeline


In [29]:
execution = pipeline.start()

print("Pipeline started ✅")
print(execution.arn)

Pipeline started ✅
arn:aws:sagemaker:ap-south-1:812224290846:pipeline/beverage-price-prediction-pipeline/execution/hzk4yrx7lcno


In [30]:
execution.wait(
    delay=20,
    max_attempts=90,
)

description = execution.describe()

print(
    "Final status:",
    description[
        "PipelineExecutionStatus"
    ]
)

Final status: Succeeded


In [31]:
steps = execution.list_steps()

for step in steps:
    print(
        step["StepName"],
        "→",
        step["StepStatus"]
    )

RegisterBeverageModel → Succeeded
BeverageProductionRefit → Succeeded
CheckModelQuality → Succeeded
BeverageModelEvaluation → Succeeded
BeverageXGBoostTraining → Succeeded
BeverageTrainTestSplit → Succeeded
BeveragePreprocessing → Succeeded


In [32]:
import boto3
import json

s3 = boto3.client("s3")

evaluation_s3_uri = (
    step_evaluate
    .properties
    .ProcessingOutputConfig
    .Outputs["evaluation"]
    .S3Output
    .S3Uri
)

# Resolve the actual processing output from the completed execution
steps = execution.list_steps()

eval_step = next(
    x for x in steps
    if x["StepName"] == "BeverageModelEvaluation"
)

processing_job_arn = (
    eval_step["Metadata"]
    ["ProcessingJob"]
    ["Arn"]
)

processing_job_name = (
    processing_job_arn.split("/")[-1]
)

sm = boto3.client("sagemaker")

desc = sm.describe_processing_job(
    ProcessingJobName=processing_job_name
)

outputs = (
    desc["ProcessingOutputConfig"]
    ["Outputs"]
)

evaluation_uri = next(
    x["S3Output"]["S3Uri"]
    for x in outputs
    if x["OutputName"] == "evaluation"
)

print("Evaluation output:", evaluation_uri)

bucket_name = evaluation_uri.replace(
    "s3://", ""
).split("/", 1)[0]

prefix = evaluation_uri.replace(
    f"s3://{bucket_name}/", ""
)

objects = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix=prefix
)

evaluation_key = next(
    obj["Key"]
    for obj in objects["Contents"]
    if obj["Key"].endswith(
        "evaluation.json"
    )
)

evaluation = json.loads(
    s3.get_object(
        Bucket=bucket_name,
        Key=evaluation_key
    )["Body"].read()
)

print(
    json.dumps(
        evaluation,
        indent=4
    )
)

Evaluation output: s3://krushang-beverage-ml-2026/pipeline/evaluation
{
    "metrics": {
        "accuracy": 0.9245560154893844,
        "macro_precision": 0.923739518150473,
        "macro_recall": 0.9234637084219806,
        "macro_f1": 0.9235755168498937,
        "weighted_f1": 0.9246687816520863,
        "ordinal_mae": 0.07544398451061557
    },
    "test_rows": 7489,
    "feature_count": 27
}


In [33]:
packages = sm.list_model_packages(
    ModelPackageGroupName=(
        "beverage-price-prediction-xgboost"
    ),
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=5,
)

print("Latest Registry packages:\n")

for pkg in packages["ModelPackageSummaryList"]:
    print(
        "Version:",
        pkg["ModelPackageVersion"],
        "| Status:",
        pkg["ModelPackageStatus"],
        "| Approval:",
        pkg["ModelApprovalStatus"],
    )

Latest Registry packages:

Version: 4 | Status: Completed | Approval: PendingManualApproval
Version: 3 | Status: Completed | Approval: Approved
Version: 2 | Status: Completed | Approval: Rejected
Version: 1 | Status: Completed | Approval: Rejected


## Pipeline Execution Result

The end-to-end SageMaker Pipeline executed successfully.

### Pipeline stages
1. Managed preprocessing
2. Stratified development/test split
3. Candidate XGBoost training
4. Untouched holdout evaluation
5. Macro-F1 quality gate
6. Full-data production refit
7. SageMaker Model Registry registration

### Automated holdout evaluation

- Accuracy: 0.9246
- Macro Precision: 0.9237
- Macro Recall: 0.9235
- Macro F1: 0.9236
- Weighted F1: 0.9247
- Ordinal MAE: 0.0754
- Test rows: 7,489
- Features: 27

The quality gate required Macro-F1 >= 0.90 and passed.

A new model package was registered as Version 4 with
`PendingManualApproval`, preserving separation between automated
quality validation and human production approval.

The currently deployed and approved production package remains Version 3.